In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@ecommercenidhi.dfs.core.windows.net/"
    )
)

In [0]:
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://bronze@ecommercenidhi.dfs.core.windows.net/customers.csv")

display(customers_df)

In [0]:
customers_df.printSchema()

In [0]:
print("Total customers:", customers_df.count())
print("Total rows:", customers_df.count())
print("Distinct rows:", customers_df.distinct().count())
from pyspark.sql.functions import col, sum

customers_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customers_df.columns
]).show()

display(customers_df)

In [0]:
customers_df.groupBy(customers_df.columns) \
    .count() \
    .filter("count > 1") \
    .show(truncate=False)

In [0]:
display(
    customers_df.filter(customers_df.city.isNull())
)

In [0]:
from pyspark.sql.functions import col, trim, lower, coalesce, lit, to_date

In [0]:
silver_customers_df = (
    customers_df
    .dropDuplicates()
    .withColumn("name", trim(col("name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", coalesce(trim(col("city")), lit("Unknown")))
    .withColumn("country", trim(col("country")))
    .withColumn("signup_date", to_date(col("signup_date")))
)

In [0]:
silver_customers_df.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
print("Bronze rows:", customers_df.count())
print("Silver rows:", silver_customers_df.count())

print("Unknown cities:")
silver_customers_df.filter(col("city") == "Unknown").show()

In [0]:
silver_path = "abfss://silver@ecommercenidhi.dfs.core.windows.net/customers"

silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
silver_path = "abfss://silver@ecommercenidhi.dfs.core.windows.net/customers"

silver_df = spark.read.format("delta").load(silver_path)

display(silver_df)

In [0]:
print("Silver row count:", silver_df.count())

silver_df.printSchema()

print("Unknown cities:")
display(
    silver_df.filter(col("city") == "Unknown")
)

In [0]:
from pyspark.sql import functions as F

silver_base = "abfss://silver@ecommercenidhi.dfs.core.windows.net"

orders = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/orders")
)

customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/customers")
)

products = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/products")
)

print("Orders:", orders.count())
print("Customers:", customers.count())
print("Products:", products.count())

In [0]:
customer_orders = (
    orders
    .join(
        F.broadcast(customers),
        "customer_id",
        "left"
    )
)

display(customer_orders.limit(10))

In [0]:
from pyspark.sql import functions as F

silver_base = "abfss://silver@ecommercenidhi.dfs.core.windows.net"
gold_base = "abfss://gold@ecommercenidhi.dfs.core.windows.net"

orders = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/orders")
)

customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/customers")
)

products = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/products")
)

sales_daily = (
    orders
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum("amount"), 2).alias("total_revenue"),
        F.round(F.avg("amount"), 2).alias("average_order_value")
    )
)

print("Orders:", orders.count())
print("Customers:", customers.count())
print("Products:", products.count())
print("Sales days:", sales_daily.count())

In [0]:
partitioned_path = (
    "abfss://gold@ecommercenidhi.dfs.core.windows.net/"
    "sales_daily_partitioned"
)

sales_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .save(partitioned_path)

In [0]:
display(
    dbutils.fs.ls(partitioned_path)
)

In [0]:
sales_daily.explain(True)